# Session 10b - CSET re-audit


**GPU T4 x2, ~3.5 h.** Re-audit on TEST with the identical command used for the base
model, so that the comparison is like for like.

**10b** audits the `mask` adapter.

The full re-audit would take about 7 h, which is close to the 10 h wall once loading
and resume overhead are counted, so it is split across **10a** and **10b**. Each half
is an independent session with its own output dataset.

In [ ]:
SESSION = "S10b CSET audit"

# ============================== CONFIG ==============================
BASE_MODEL  = "Qwen/Qwen2.5-VL-3B-Instruct"
VARIANTS    = ['mask']   # "" = the untuned base model
QUANT       = "fp16"
MAX_PIXELS  = 384 * 384
SPLIT       = "test"
SAMPLES     = 600
SEED        = 0
PROMPT_VARIANTS = 5
BUDGET_PER_VARIANT_MIN = 150

In [ ]:
# ---------------------------------------------------------------- BOOT
# Locates the code dataset wherever it is mounted, puts it on sys.path,
# prints the attached inputs, and records provenance.  The search is by
# file name, so the dataset's mount name does not matter.
import os, sys, subprocess, json, time

def _find_code():
    for root in ("/kaggle/input", "."):
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, files in os.walk(root):
            dirnames[:] = [d for d in dirnames if not d.startswith(".")]
            if os.path.basename(dirpath) == "ccaudit" and "kaggle_utils.py" in files:
                return os.path.dirname(dirpath)
    raise FileNotFoundError(
        "Could not find the ccaudit package.\n"
        "Add Input -> your code dataset (<your-code-dataset>), and check that "
        "the preview shows ccaudit/kaggle_utils.py at the top level.")

CODE = _find_code()
if CODE not in sys.path:
    sys.path.insert(0, CODE)
# Child processes do not inherit sys.path.  Every `python -m ccaudit.<module>`
# below runs as a subprocess, so the code directory must be on PYTHONPATH.
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
from ccaudit import kaggle_utils as KU
from ccaudit import common as C

OUT = KU.work_dir("audit")
TMP = KU.temp_dir()
os.environ["HF_HOME"] = KU.temp_dir("hf")          # model weights stay out of /kaggle/working
os.environ["TOKENIZERS_PARALLELISM"] = "false"
KU.session_header(SESSION, OUT)
print("code:", CODE)

In [ ]:
# ------------------------------------------------------- SELF TEST (always)
# The self-test suite runs in under a minute and needs no dataset.  Each check
# corresponds to a failure mode that would produce plausible-looking but
# incorrect numbers, so a failure here invalidates everything that follows.
rc = KU.sh(f"{sys.executable} {CODE}/scripts/selftest.py", check=False)
if rc != 0:
    raise SystemExit("SELF TEST FAILED -- inspect the failures above before proceeding.")

In [ ]:
KU.pip_install("transformers accelerate qwen-vl-utils peft bitsandbytes")
KU.gpu_report()
INDEX = KU.find_parsed_index()
recs, meta = C.load_index(INDEX)
VOCAB = meta.get("vocab", "face8")
n_gpu = max(1, KU.n_gpus())

# Locate the adapters produced by Session 9.
import glob
ADAPTERS = {}
for root in ("/kaggle/input", KU.work_dir()):
    for p in glob.glob(f"{root}/**/cset/*/adapter_config.json", recursive=True):
        ADAPTERS[os.path.basename(os.path.dirname(p))] = os.path.dirname(p)
print("adapters found:", ADAPTERS or "(none -- attach `cca-s9-cset`)")
# Seed the cache from earlier re-audit runs and from the Session 5 main run:
# the base-model command here is identical to Session 5's (same detector,
# split, samples, seed and settings), and cache keys are content-addressed,
# so an attached `cca-s5-vlm-*` dataset lets the base re-audit rebuild from
# cache instead of re-running the model.
RESUME = ",".join(d for d in KU.find_run_dirs()
                  if "run_cset" in d or "run_vlm" in d)

In [ ]:
# ==================== RE-AUDIT ====================
for v in VARIANTS:
    if v:
        adir = ADAPTERS.get(v)
        if not adir:
            print(f"skipping {v}: no adapter attached")
            continue
        det, tag = f"qwen25vl:{BASE_MODEL}:lora={adir}", f"cset_{v}"
    else:
        det, tag = f"qwen25vl:{BASE_MODEL}", "cset_base"
    cmds, envs, logs = [], [], []
    for i in range(n_gpu):
        logs.append(f"{OUT}/logs/{tag}_{i}.log")
        envs.append({"CUDA_VISIBLE_DEVICES": str(i)})
        cmds.append(
            f'{sys.executable} -m ccaudit.m5_runner --index "{INDEX}" '
            f'--detector "{det}" --out "{OUT}/run_cset" --tag {tag} '
            f'--split {SPLIT} --limit-samples {SAMPLES} --seed {SEED} '
            f'--shard {i}/{n_gpu} --device cuda --quant {QUANT} '
            f'--max-pixels {MAX_PIXELS} --prompt-variants {PROMPT_VARIANTS} '
            f'--splice-floor --vocab {VOCAB} '
            f'--time-budget-min {BUDGET_PER_VARIANT_MIN}'
            + (f' --resume-from "{RESUME}"' if RESUME else ""))
    print(C.banner(tag))
    KU.run_parallel(cmds, envs=envs, logs=logs, check=False, poll_sec=60)

In [ ]:
# ==================== INTERIM COMPARISON ====================
KU.sh(f'{sys.executable} -m ccaudit.m6_metrics --raw "{OUT}/run_cset" '
      f'--out "{OUT}/metrics_cset" --split {SPLIT} --vocab {VOCAB}', check=False)
print(f"{'variant':16s}{'AUC':>8}{'FS':>10}{'CR-prior':>10}")
for r in C.load_json(f"{OUT}/metrics_cset/metrics.json", {}).get("results", []):
    print(f"{str(r.get('tag')):16s}{r.get('AUC',float('nan')):>8.3f}"
          f"{r.get('FS',float('nan')):>10.4f}"
          f"{r.get('CR_minus_prior',float('nan')):>10.3f}")
print("\nA drop in AUC of more than 0.01 relative to the base model is "
      "reported as a detection trade-off alongside any change in FS or CR.")

In [ ]:
NEXT_STEP = """1. Output tab -> New Dataset -> `cca-s10b-cset`.
2. Run the other half (10a) if not already done.
3. Attach both to Session 11 for the final table."""

# ----------------------------------------------------------- WRAP UP
KU.disk_report()
print(C.banner("NEXT STEP"))
print(NEXT_STEP)